## Cell 1: Import Libraries

In [2]:
import yfinance as yf
import pandas as pd
import numpy as np
import os

## Cell 2: Loading, Cleaning, and Saving data for the tickers in our universe

In [3]:
etfs = [
    "SPY",  # In order to enforce market neutrality
    "XLF",  # Financials
    "XLE",  # Energy
    "XLK",  # Technology
    "XLI",  # Industrials
    "XLP",  # Consumer Staples
    "XLY",  # Consumer Discretionary
    "XLV",  # Health Care
    "XLU",  # Utilities
    "IYT",  # Transportation
    "XRT",  # Retail
    "SMH",  # Semiconductors
    "IYR",  # Real Estate
    "IGV",  # Software/Internet
    "XLB",  # Materials
    "VOX",  # Communication Services
]
START_DATE = "2018-01-01"
END_DATE = "2023-12-31"
WIKI_SP500_2018_URL = "https://en.wikipedia.org/w/index.php?title=List_of_S%26P_500_companies&oldid=820572003"
WIKI_SP400_2018_URL = "https://en.wikipedia.org/w/index.php?title=List_of_S%26P_400_companies&oldid=818353762"

# S&P 500+400 tickers (grab from Wikipedia)
sp500 = pd.read_html(
    WIKI_SP500_2018_URL,
    storage_options={"User-Agent": "Mozilla/5.0"},
)[0]
sp400 = pd.read_html(
    WIKI_SP400_2018_URL,
    storage_options={"User-Agent": "Mozilla/5.0"},
)[0]
sp400.rename(columns={"Ticker Symbol": "Symbol"}, inplace=True)
sp500.rename(columns={"Ticker symbol": "Symbol"}, inplace=True)

# yfinance uses '-' not '.' in tickers
sp500["Symbol"] = sp500["Symbol"].str.replace(".", "-", regex=False)
sp400["Symbol"] = sp400["Symbol"].str.replace(".", "-", regex=False)

stock_tickers = list(set(sp500["Symbol"].tolist() + sp400["Symbol"].tolist()))

all_tickers = stock_tickers + etfs

print(
    f"Requesting data for {len(stock_tickers)} stock and {len(etfs)} etf tickers from Yahoo Finance..."
)

raw_data = yf.download(
    tickers=all_tickers,
    start=START_DATE,
    end=END_DATE,
    auto_adjust=True,
    threads=True,
    progress=False,
)
prices = raw_data["Close"]
volume = raw_data["Volume"]

print(f"Downloaded: {len(prices.columns)} / {len(all_tickers)}")

# Pre-cleaning diagnostics
print(f"\n--- Raw Price Data ---")
print(f"Total NaN:         {prices.isna().sum().sum()}")
print(f"Tickers with NaN:  {(prices.isna().sum() > 0).sum()}")
print(
    f"Worst ticker:      {prices.isna().sum().idxmax()} ({prices.isna().sum().max()} NaN)"
)

# Keep stocks with atleast 95% of data
prices = prices.dropna(axis=1, thresh=int(0.95 * len(prices)))
print(f"Dropped (<95%):    {len(all_tickers) - len(prices.columns)} tickers")
prices = prices.ffill()
remaining_nan = prices.isna().sum().sum()
print(f"NaN after ffill:   {remaining_nan}")
prices = prices.dropna()
print(f"Rows dropped:      {len(raw_data['Close']) - len(prices)}")

# Calculate simple returns
returns = prices.pct_change().dropna()

print(f"\n--- Returns Quality ---")
print(f"Shape:             {returns.shape}")
print(f"NaN count:         {returns.isna().sum().sum()}")
print(f"Inf count:         {np.isinf(returns.values).sum()}")
print(f"Max return:        {returns.max().max():.4f} ({returns.max().idxmax()})")
print(f"Min return:        {returns.min().min():.4f} ({returns.min().idxmin()})")
print(f"Mean abs return:   {returns.abs().mean().mean():.6f}")

volume = volume[prices.columns]
volume = volume.loc[returns.index]
volume[volume == 0] = np.nan
volume = volume.ffill().bfill()

print(f"Remaining: {len(prices.columns)} / {len(all_tickers)}")

# Separate etfs from sp500
etf_cols = [c for c in prices.columns if c in etfs]
stock_cols = [c for c in prices.columns if c not in etfs]

etf_prices = prices[etf_cols]
stock_prices = prices[stock_cols]

etf_returns = returns[etf_cols]
stock_returns = returns[stock_cols]
spy_returns = returns["SPY"]

stock_volume = volume[stock_cols]

# Add to data directory
os.makedirs("../data", exist_ok=True)
spy_returns.to_csv("../data/raw/spy_returns.csv")
etf_prices.to_csv("../data/raw/etf_prices.csv")
stock_prices.to_csv("../data/raw/stock_prices.csv")
etf_returns.to_csv("../data/raw/etf_returns.csv")
stock_returns.to_csv("../data/raw/stock_returns.csv")
stock_volume.to_csv("../data/raw/stock_volume.csv")

print("Successfully saved prices and returns data.")
print(f"SPY returns:   {spy_returns.shape}")
print(f"ETF prices:   {etf_prices.shape}")
print(f"Stock prices: {stock_prices.shape}")
print(f"ETF returns:   {etf_returns.shape}")
print(f"Stock returns: {stock_returns.shape}")
print(f"Stock Volume: {stock_volume.shape}")

Requesting data for 904 stock and 16 etf tickers from Yahoo Finance...


$UTX: possibly delisted; no timezone found
$FLIR: possibly delisted; no timezone found
$JW-A: possibly delisted; no timezone found
$NCR: possibly delisted; no timezone found
$PLT: possibly delisted; no price data found  (1d 2018-01-01 -> 2023-12-31) (Yahoo error = "Data doesn't exist for startDate = 1514782800, endDate = 1703998800")
$ATVI: possibly delisted; no timezone found
$DNKN: possibly delisted; no timezone found
$ESL: possibly delisted; no timezone found
$RTN: possibly delisted; no timezone found
$PACW: possibly delisted; no timezone found
$JEC: possibly delisted; no timezone found
$ARNC: possibly delisted; no timezone found
$MDSO: possibly delisted; no timezone found
$GGP: possibly delisted; no price data found  (1d 2018-01-01 -> 2023-12-31)
$INT: possibly delisted; no timezone found
$DISH: possibly delisted; no timezone found
$DRQ: possibly delisted; no timezone found
$TMK: possibly delisted; no timezone found
$THS: possibly delisted; no price data found  (1d 2018-01-01 -> 20

Downloaded: 920 / 920

--- Raw Price Data ---
Total NaN:         323813
Tickers with NaN:  221
Worst ticker:      AAN (1509 NaN)
Dropped (<95%):    220 tickers
NaN after ffill:   0
Rows dropped:      0

--- Returns Quality ---
Shape:             (1508, 700)
NaN count:         0
Inf count:         0
Max return:        1.3484 (GME)
Min return:        -0.6412 (MTDR)
Mean abs return:   0.015636
Remaining: 700 / 920
Successfully saved prices and returns data.
SPY returns:   (1508,)
ETF prices:   (1509, 16)
Stock prices: (1509, 684)
ETF returns:   (1508, 16)
Stock returns: (1508, 684)
Stock Volume: (1508, 684)


## Cell 3: Creating Metadata Table

In [17]:
sp500_meta = sp500[["Symbol", "Security", "GICS Sector", "GICS Sub Industry"]].copy()
sp400_meta = sp400[
    ["Symbol", "Company", "GICS Economic Sector", "GICS Sub-Industry"]
].copy()
sp400_meta.columns = ["Symbol", "Security", "GICS Sector", "GICS Sub Industry"]

metadata = pd.concat([sp500_meta, sp400_meta]).drop_duplicates(subset="Symbol")
metadata = metadata[metadata["Symbol"].isin(stock_returns.columns)]


def map_etf_and_sector(row):
    sector = row["GICS Sector"]
    sub_ind = row["GICS Sub Industry"]

    # 1. Specialized Tech (IGV & SMH)
    if sub_ind in [
        "Application Software",
        "Systems Software",
        "Home Entertainment Software",
        "Internet Services & Infrastructure",
    ]:
        return pd.Series(["Software/Internet", "IGV"])
    if sub_ind in [
        "Semiconductors",
        "Semiconductor Materials & Equipment",
        "Semiconductor Equipment",
    ]:
        return pd.Series(["Semiconductors", "SMH"])

    # 2. Specialized Industrials/Consumer (IYT & XRT)
    if sub_ind in [
        "Air Freight & Logistics",
        "Railroads",
        "Rail Transportation",
        "Airlines",
        "Passenger Airlines",
        "Trucking",
        "Cargo Ground Transportation",
    ]:
        return pd.Series(["Transportation", "IYT"])
    if sub_ind in [
        "Broadline Retail",
        "Automotive Retail",
        "Computer & Electronics Retail",
        "Home Improvement Retail",
        "Apparel Retail",
        "Specialty Stores",
        "Department Stores",
    ]:
        return pd.Series(["Retail", "XRT"])

    # 3. Broad GICS Sector Mapping
    mapping = {
        "Financials": ["Financials", "XLF"],
        "Energy": ["Energy", "XLE"],
        "Information Technology": ["Information Technology", "XLK"],
        "Industrials": ["Industrials", "XLI"],
        "Consumer Staples": ["Consumer Staples", "XLP"],
        "Consumer Discretionary": ["Consumer Discretionary", "XLY"],
        "Health Care": ["Health Care", "XLV"],
        "Utilities": ["Utilities", "XLU"],
        "Real Estate": ["Real Estate", "IYR"],
        "Materials": ["Materials", "XLB"],
        "Communication Services": ["Communication Services", "XLC"],  # or VOX
    }
    if sector in mapping:
        return pd.Series(mapping[sector])

    return pd.Series([None, None])


# Map to our universe of sectors/etfs
metadata[["sector", "etf"]] = metadata.apply(map_etf_and_sector, axis=1)

metadata = metadata.drop(columns=["GICS Sector", "GICS Sub Industry"])

metadata.set_index("Symbol", inplace=True)

# Save metadata
metadata.to_csv("../data/stock_meta.csv", index=True)

# Print the count by readable Sector Name
print(f"\n--- Metadata Summary ---")
print(f"Total stocks:   {len(metadata)}")
print(f"Total sectors:  {metadata['sector'].nunique()}")
print(f"Unmapped:       {metadata['sector'].isna().sum()}")

print("\n--- Number of Stocks per Target Sector ---")
display(metadata["sector"].value_counts().to_frame(name="Count"))



--- Metadata Summary ---
Total stocks:   686
Total sectors:  14
Unmapped:       3

--- Number of Stocks per Target Sector ---


,Count
sector,
Financials,98
Industrials,85
Consumer Discretionary,84
Health Care,64
Information Technology,63
Real Estate,50
Materials,45
Consumer Staples,45
Utilities,40
